# MSM to map a kinetic model onto MD trajectories

In this notebook, we introduce  a typical workflow to analyse 
an MD trajectory. 

The typical steps are :
1. Choose some features to analyze (positions, angles, distances...). 
2. Reduction of dimensionality (PCA, TICA, VAMP, machine learning,....)
3. Clustering to get a set of $k$ microstates ($k$ can be as large as 1000).
4. Calculate an MSM on the $k$ microstates using a timelag $\tau$.  Validate the MSM using timelag convergence.
5. Make a coarse-graining of microstates into  a few ($n$) kinetic metastable states (PCCA procedure). Test the markovianity using CK test on $n$ microstates.
6. Analyse the kinetics of between these $n$ metastable states.


In practice, often the steps 2 to 5 are reitterated until all the parameters chosen at the different steps converge towards a reasonable MSM.

Here, some of the "good" parameters will be given, other will be tested.
A measure of the quality of a feature-set or of a model is called the VAMPS-score.



In [ ]:

%matplotlib inline
import os
os.environ["OMP_NUM_THREADS"] = "4"

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import mdshare
import pyemma
from pyemma.util.contexts import settings
import pexpect
import random
from itertools import product

## 0. VAMP-score

Before we begin, we shall define the VAMP-score.
In order to choose the parameters of our MSM models, we want a quantitative criterium that tell us whether the
model is "interesting". We say it will be interesting if it contains several slow dynamics.

 The *VAMP-score*  measures the *kinetic variance* contained in the observables. It is a measure of the timescales of the feature variations during the simulations. The higher the score, the better the feature capture a **slow** dynamics. 

The minimum value of this score is $1$, which corresponds to the invariant measure or the value obtained if the features selected are at equilibrium. There is no dynamics at all ! 

VAMP is based on the Canonical Correlation Analysis (CCA)
method for time-series data and also known as Time-lagged
Canonical Correlation Analysis (TCCA). 

**Reminder about the cross correlation function between a pair of variables $(x, y)$**.

If we have $N$ values of the variable $x$, noted $\{x_i\}_{i=1,..,N}$, and the same for $y$. Then the cross correlation is :
$$
C_0(x,y) = \frac{\sum_{i=1}^N (x_i-\hat{x})(y_i-\hat{y}) }{N \sigma_x \sigma_y}
$$
where $\hat{x}$ and $\sigma_x$ are the mean and variance of the variable $x$.

CCA find the linear combinations $\mathcal{r} = r_x x + r_y y$ of variables $x$ and $y$ that maximizes the cross correlation functions. These variables are sometimes called **collective variables**.

One can also look at **time-lagged cross correlation matrices**, ie the cross correlation of the variable $x$ with the variable $y$ with a time shift of $\tau$ steps.
$$
C_\tau(x,y) = \frac{\sum_{i=1}^{N-\tau} (x_i-\hat{x})(y_{i+\tau}-\hat{y}) }{(N-\tau) \sigma_x \sigma_y}
$$

**The VAMP-2 score** is the sum of the largest diagonal elements of that time-lagged cross correlation matrix raised to a power of 2. 

**The higher the value of this score, the better the collective variables
are  conserving the slowest kinetic modes, the most interesting they are**

In [ ]:
# definition of the function to calculate the VAMP2 score
# the collective variables are obtained from a training subset of the data
# the eigenvalues are obtained from the rest (test set) of the data
# To compute an error bar, the splitting into training/validating is done several times.

def score_cv(data, dim, lag, number_of_splits=2, validation_fraction=0.5):
    """Compute a cross-validated VAMP2 score.
    
    We randomly split the list of independent trajectories into
    a training and a validation set, compute the VAMP2 score,
    and repeat this process several times.
    
    Parameters
    ----------
    data : list of numpy.ndarrays
        The input data.
    dim : int
        Number of processes to score; equivalent to the dimension
        after projecting the data with VAMP2.
    lag : int
        Lag time for the VAMP2 scoring.
    number_of_splits : int, optional, default=10
        How often do we repeat the splitting and score calculation.
    validation_fraction : int, optional, default=0.5
        Fraction of trajectories which should go into the validation
        set during a split.
    """
    # we temporarily suppress very short-lived progress bars
    with pyemma.util.contexts.settings(show_progress_bars=False):
        nval = int(len(data) * validation_fraction)
        scores = np.zeros(number_of_splits)
        for n in range(number_of_splits):
            ival = np.random.choice(len(data), size=nval, replace=False)
            vamp = pyemma.coordinates.vamp(
                [d for i, d in enumerate(data) if i not in ival], lag=lag, dim=dim)
            scores[n] = vamp.score([d for i, d in enumerate(data) if i in ival])
    return scores


**We shall again use the pentapeptide case you have looked at for the first practical**

In the next cell, you download the data, and look at the first trajectory using vmd.

In [ ]:
WD = "./"
datadir = WD + "/data"
pdb = mdshare.fetch('pentapeptide-impl-solv.pdb', working_directory=datadir,show_progress=False)
files = mdshare.fetch('pentapeptide-*-500ns-impl-solv.xtc', working_directory=datadir,show_progress=False)

print(f"topology file stored in {pdb}")
print(f"trajectory files are {files}")

# visualise the peptide trajectory using vmd
Vizualize = True
if Vizualize : 
    vmd = pexpect.spawn("/usr/local/bin/vmd-1.9.4a57") # Ou "/chemin/vers/le/programme"
    vmd.expect("vmd > ") # Le paramètre est le prompt du programme
    vmd.sendline(f"mol load pdb {pdb} xtc {files[0]}")
    vmd.expect("vmd > ")



<font color='red'>

**QUESTIONS**  
</font>

1. Look at one trajectory using vmd. Can you regognize some metastable states  with the eyes ? How many ?
2. How many conformations are in a single trajectory of 500ns ? What is the timestep between the different frames of the trajectory. 
3.In the next cell, define the variable `time_step_traj_ns` that represents this timestep, in nanosecond.

In [ ]:
# the step in the trajectory is 0.1ns
time_step_traj_ns = 0.1
#time_step_traj_ns = TOCOMPLETE

## 1. Feature selection

In the following bunches of observables  are calculated for  all torsion angles of the backbone, 
We eliminate the side-chain dynamics, since we are interested in the slow kinetics of the peptide conformations.



In [ ]:

torsions_feat = pyemma.coordinates.featurizer(pdb)
torsions_feat.add_backbone_torsions(cossin=True, periodic=False)
torsions_data = pyemma.coordinates.load(files, features=torsions_feat)
labels = ['backbone\ntorsions']



First, we shall look at the time evolution of a few torsion angles for a chosen trajectory.


In [ ]:
# choose a trajectory
chosen_traj = 0
print(f"Number of timesteps in {np.shape(torsions_data[chosen_traj])[0]}")
print(f"Number of observables at each timestep {np.shape(torsions_data[chosen_traj])[1]}")

# plot the torsion data as a function of time 
# for the chosen trajectory
# and the four first torsion angles 

data_to_plot = np.array(torsions_data[chosen_traj].T)[0:4]
print(np.shape(data_to_plot))

fig, axes = plt.subplots(4, 1, figsize=(12, 5), sharex=True)
x = time_step_traj_ns * np.arange(len(data_to_plot[0]))

for i, (ax, tor) in enumerate(zip(axes.flat,data_to_plot)):
    ax.plot(x, tor)
    ax.set_title(torsions_feat.describe()[i])
axes[-1].set_xlabel('time / ns')
#axes[-1].set_xlabel('time /  UNITS TOCOMPLETE')


fig.tight_layout()

<font color='red'>

**QUESTIONS**  
</font>
1.  Can you see jumps between a few metastable states for all the torsion angles ? 

## 2. Dimensionality reduction

We shall use the TICA dimensionality reduction. It is again aiming at finding the linear combinaison of features that maximizes the timescale of variation of the features.

Given $C_0$  and $C_\tau$ the mean-free covariance and time-lagged covariance matrix, 
calculated for a given lag time $\tau$,
the new collective variables $r_i$ are the solutions of the eigenvalue problem :
$$
C_\tau r_i = C_0 \lambda_i r_i.
$$
where $r_i$ are the independant collective varibales, and $\lambda_i(\tau) $ are the respective eigenvalues. The eigenvalues are related to the relaxation timescale by :
$$
t_i = -\frac{\tau}{\ln(\lambda_i)}
$$

**We use the eigenvalues to make a dimensionality reduction**.
We use only the eigenvectors with the largest eigenvalues, ie the largest relaxation timescales, and use them as collective variables. 

It means that we have two  choose two parameters for the dimensionality reduction
1. a $\tau$ for the time-lagged covariance matrix.  We do not know a priori the best choice.
2. the number of dimensions. We want the smallest number that is able to represent the dynamics of the system. 



<font color='red'>

**QUESTIONS**  
</font>
1.  In the next cell, we shall look at the VAMP-score for different dimensionality reduction.
Since we do not know which time $\tau$ is best yet, we compare for different lags $\tau$.

Complete the lists of lag times (integer, in timesteps) and dimensions. 




In [ ]:
lags = [1, 2, 5, 10, 20]
dims = [i + 1 for i in range(10)]

#lags = [ TOCOMPLETE, ]
#dims = [ TOCOMPLETE, ]

fig, ax = plt.subplots()
for i, lag in enumerate(lags):
    scores_ = np.array([score_cv(torsions_data, dim, lag)
                        for dim in dims])
    scores = np.mean(scores_, axis=1)
    errors = np.std(scores_, axis=1, ddof=1)
    color = 'C{}'.format(i)
    ax.fill_between(dims, scores - errors, scores + errors, alpha=0.3, facecolor=color)
    ax.plot(dims, scores, '--o', color=color, label='lag={:.1f}ns'.format(lag * time_step_traj_ns))
ax.legend()
ax.set_xlabel('number of dimensions')
ax.set_ylabel('VAMP2 score')
fig.tight_layout()

We observe that increasing the number of dimensions leads to better results until a convergence.
But a model with more dimensions is more difficult to handle in a statistical point of view, more complex to understand. So the smallest number of dimension is wished. 

<font color='red'>

**QUESTIONS**  
</font>

1. For a lag time of 0.1ns, how many dimensions are necessary to represent the dynamics of the backbone torsion angle ?
2. And for a lag time of 0.5ns to 2.0 ns ? Why is it lower than for 0.1ns ?

In the following, we shall  make a larger time-coarse-graining, 
and use $\tau$=0.5ns for the dimensionaly reduction.

In the next cell, define the two variables for the dimensionality reduction.
1. `dim_reduction_chosen_dim`
2. `dim_reduction_chosen_lag`. *Remember to define it an integer that represent the multiple of the time-step of the trajectory.*
 

In [ ]:
dim_reduction_chosen_dim = 4
dim_reduction_chosen_lag = 5
#dim_reduction_chosen_dim = TOCOMPLETE
#dim_reduction_chosen_lag = TOCOMPLETE

tica = pyemma.coordinates.tica(torsions_data, lag=dim_reduction_chosen_lag, dim = dim_reduction_chosen_dim)
tica_output = tica.get_output()
tica_concatenated = np.concatenate(tica_output)



In the next cell, we shall  plot the coefficients 
of each torsion angle in the different collective variables, ie. the TICA projections.
Look at the coefficients.

<font color='red'>

**QUESTIONS**  
</font>

1. What are the two torsion angles that participate most to the first eigenvalue ? 


In [ ]:
print(tica.describe())

# DESCRIPTION OF THE COMPONENTS OF THE FEATURES INTO THE TICA COORDINATES

fig, ax = plt.subplots(figsize=(20, 0.5*torsions_feat.dimension()))
i = ax.imshow(tica.feature_TIC_correlation, cmap='bwr')

ax.set_xticks(range(tica.dimension()))
ax.set_xlabel('TICA')
ax.set_yticks(range(torsions_feat.dimension()))
ax.set_yticklabels(torsions_feat.describe())
ax.set_ylabel('input feature')

fig.colorbar(i);
#fig.savefig(f'{PICTDIR}/PCA_feature_coefficients.png')


*Technical note : The above variable `tica_concatenated` 
is a concatenation of independent trajectories 
and as such **not suitable** for any kind of analysis which **evaluates transitions**.
We use this variable purely for visualization purposes which require concatenated data.
Throughout the notebook, we will use the `_concatenated` 
postfix to denote data which is not safe to run a TICA or MSM estimation on.*


Let’s have a look at the first trajectory and what it looks like in the space of the first four TICA components. 

In [ ]:
# figure of the time evolution of the new collective variables 
# defined from the TICA projection

fig, axes = plt.subplots(4, 1, figsize=(12, 5), sharex=True)
x = time_step_traj_ns * np.arange(tica_output[0].shape[0])
for i, (ax, tic) in enumerate(zip(axes.flat, tica_output[0].T)):
    ax.plot(x, tic)
    ax.set_ylabel('IC {}'.format(i + 1))
axes[-1].set_xlabel('time / ns')
fig.tight_layout()

<font color='red'>

**QUESTIONS**  
</font>

1. Do you observe  short jumps between different metasable states in the new "collective variables" ?




In the next cell, we visualise  the marginal and joint distributions of our TICA components by simple histograming




The projection yields defined clusters of high density, which are most likely to be identified as metastable basins. 




In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
pyemma.plots.plot_feature_histograms(
    tica_concatenated,
    ax=axes[0],
    feature_labels=['IC1', 'IC2', 'IC3', 'IC4'],
    ylog=True)
pyemma.plots.plot_density(*tica_concatenated[:, :2].T, ax=axes[1], logscale=True)
axes[1].set_xlabel('IC 1')
axes[1].set_ylabel('IC 2')
fig.tight_layout()



## 3. Discretization of the (reduced-dimension) phase-space into microstates

The TICA coordinates will now be clustered into a number of discrete states using the $k$-means algorithm. It requires as input the desired number of clusters.

To guess a sufficient number of clusters,
we use the VAMP-2 score again, applied this time to 
the Markov models obtained with different numbers of clusters.

This approach requires us to guess a MSM lag time.
In a first guess, we set it to the TICA lag time of $5$ steps (or $0.5$ ns).

Since the clustering algorithm is stochastic,
we conduct multiple rounds of discretization at each number of cluster centers (here, 5 times).

<font color='red'>

**QUESTIONS**  
</font>

In the next cell, complete the variables 
1. `n_clustercenters` should contain a list of cluster numbers $k$. For example [5,100].
2. `n_repetition` should contain an integer with the number of repetition for calculation of the error on the VAMP-score.
3. `guess_MSM_lag_time` should contain an integer that define the MSM-lagtime for the calculation of the VAMP-score.  As previously, it is in steps of the trajectory.


In [ ]:
n_clustercenters = [5, 10, 30, 75, 200, 450]
n_repetition = 5
guess_MSM_lag_time = 5

#n_clustercenters = TOCOMPLETE
#n_repetition = TOCOMPLETE
#guess_lag_time = TOCOMPLETE

scores = np.zeros((len(n_clustercenters), n_repetition))

for n, k in enumerate(n_clustercenters):
    for m in range(n_repetition):
        with pyemma.util.contexts.settings(show_progress_bars=False):
            _cl = pyemma.coordinates.cluster_kmeans(
                tica_output, k=k, max_iter=50, stride=50)
            _msm = pyemma.msm.estimate_markov_model(_cl.dtrajs, lag = guess_MSM_lag_time )
            scores[n, m] = _msm.score_cv(
                _cl.dtrajs, n=1, score_method='VAMP2', score_k=min(10, k))

fig, ax = plt.subplots()
lower, upper = pyemma.util.statistics.confidence_interval(scores.T.tolist(), conf=0.9)
ax.fill_between(n_clustercenters, lower, upper, alpha=0.3)
ax.plot(n_clustercenters, np.mean(scores, axis=1), '-o')
ax.semilogx()
ax.set_xlabel('number of cluster centers')
ax.set_ylabel('VAMP-2 score')
fig.tight_layout()


From this curve, we can choose a number of microstates.
The smallest is best to have better jump statistics.
The higher is best to have a finer description of the kinetic fluxes.

<font color='red'>

**QUESTIONS**  
</font>

1. What seems a reasonable number of microstates ? 

In the next cell, complete the variable `n_chosen_microstates` with a reasonable number. 
Then make the clustering and observe the clusters on the projection on the 2 first dimensions.


In [ ]:
# Make the clustering

#n_chosen_microstates = TOCOMPLETE
n_chosen_microstates = 100

cluster = pyemma.coordinates.cluster_kmeans(
    tica_output, k=n_chosen_microstates, max_iter=50, stride=10, fixed_seed=1)
dtrajs_concatenated = np.concatenate(cluster.dtrajs)


# plot the clusters on the two first TICA dimensions

fig, ax = plt.subplots(figsize=(4, 4))
pyemma.plots.plot_density(
    *tica_concatenated[:, [0,1]].T, ax=ax, cbar=False, alpha=0.3)
ax.scatter(*cluster.clustercenters[:, [0,1]].T, s=5, c='C1')
ax.set_xlabel('IC 1')
ax.set_ylabel('IC 2')
fig.tight_layout()




##  4.MSM estimation and validation


In [ ]:

chosen_lag = 10

msm = pyemma.msm.bayesian_markov_model(cluster.dtrajs, lag=chosen_lag, dt_traj='0.1 ns')
print('fraction of states used = {:.2f}'.format(msm.active_state_fraction))
print('fraction of counts used = {:.2f}'.format(msm.active_count_fraction))


### MSM spectral analysis (eigenvalues and eigenvectors)


### Implied timescales

The first validation that is usually done when estimating a Markov model is the estimation of implied timescales (ITS) $t_i$. They are computed from the eigenvalues $\lambda_i$ of the Markov transition matrix by

$$ t_i = \frac{-\tau}{\ln\left|\lambda_i(\tau)\right|} $$ 

with $\tau$ being the lag time.
PyEMMA sorts the implied timescales (and their corresponding eigenfunctions) in descending order.




In [ ]:

nits = 15
timescales_mean = msm.sample_mean('timescales', k=nits)
timescales_std = msm.sample_std('timescales', k=nits)


fig, axes = plt.subplots(1, 1 , figsize=(6, 4))

axes.errorbar(
    range(1, nits + 1),
    timescales_mean, 
    yerr=timescales_std, 
    fmt='.', markersize=10, label = f'lag = {chosen_lag} steps')
axes.set_xticks(range(1, nits + 1))
axes.grid(True, axis='x', linestyle=':') 
axes.axhline(msm.lag * 0.1, lw=1.5, color='k')
axes.axhspan(0, msm.lag * 0.1, alpha=0.3, color='k')
axes.set_xlabel('implied timescale index')
axes.set_ylabel('implied timescales / ns')
axes.legend()
fig.tight_layout()

Note that the ITS $t_i$ approximates the decorrelation time of the $i$-th process;
it should be independent of the model's hyper-parameter $\tau$.
Thus, $\tau$ is large enough, if we increase the lag time $\tau$, 
the its should not change much.

In the following cell, we increase the lag, and check whether the implied timescale remain approximatively constant.

In [ ]:

multiply_lags = [2,4,6]
k_msm = []
for k in multiply_lags : 
    k_msm.append(pyemma.msm.bayesian_markov_model(cluster.dtrajs, lag=chosen_lag*k, dt_traj='0.1 ns'))


if False :
    its = pyemma.msm.its(cluster.dtrajs, lags=10, nits=10, errors=None, only_timescales=True)
    pyemma.plots.plot_implied_timescales(its, units='ns', dt=0.1)




In [ ]:


fig, axes = plt.subplots(1, 1 , figsize=(6, 4))

axes.errorbar(
    range(1, nits + 1),
    timescales_mean, 
    yerr=timescales_std, 
    fmt='.', markersize=10, label = f'lag = {chosen_lag} steps')

for i,k in enumerate(multiply_lags) : 
    k_timescales_mean = k_msm[i].sample_mean('timescales', k=nits)
    k_timescales_std = k_msm[i].sample_std('timescales', k=nits)

    axes.errorbar(
        range(1, nits + 1),
        k_timescales_mean, 
        yerr=k_timescales_std, 
        fmt='.', markersize=10, label = f'lag = {chosen_lag*k} steps')


axes.set_xticks(range(1, nits + 1))
axes.grid(True, axis='x', linestyle=':') 
axes.axhline(msm.lag * 0.1, lw=1.5, color='k')
axes.axhspan(0, msm.lag * 0.1, alpha=0.3, color='k')
axes.set_xlabel('implied timescale index')
axes.set_ylabel('implied timescales / ns')
axes.legend()
fig.tight_layout()

Analyzing the stationary distribution and the free energy computed over the first two TICA coordinates.

The stationary distribution, $\pi$, is stored in `msm.pi`.

We compute the free energy landscape by re-weighting the trajectory frames with stationary probabilities from the MSM (returned by `msm.trajectory_weights()`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
pyemma.plots.plot_contour(
    *tica_concatenated[:, :2].T,
    msm.pi[dtrajs_concatenated],
    ax=axes[0],
    mask=True,
    cbar_label='stationary distribution')
pyemma.plots.plot_free_energy(
    *tica_concatenated[:, :2].T,
    weights=np.concatenate(msm.trajectory_weights()),
    ax=axes[1],
    legacy=False)
for ax in axes.flat:
    ax.set_xlabel('IC 1')
axes[0].set_ylabel('IC 2')
axes[0].set_title('Stationary distribution', fontweight='bold')
axes[1].set_title('Reweighted free energy surface', fontweight='bold')
fig.tight_layout()

The eigenvectors corresponding to the slowest processes (largest implied timescales) contain information about what configurational changes are happening on which timescales.
We analyze the slowest processes by inspecting the value of the first four eigenfunctions projected on two the first TICA coordinates.
As the first right eigenvector corresponds to the stationary process (equilibrium), it is constant at $1$.

In [ ]:
eigvec = msm.eigenvectors_right()
print('The first eigenvector is one: {} (min={}, max={})'.format(
    np.allclose(eigvec[:, 0], 1, atol=1e-15), eigvec[:, 0].min(), eigvec[:, 0].max()))

fig, axes = plt.subplots(1, 4, figsize=(15, 3), sharex=True, sharey=True)
for i, ax in enumerate(axes.flat):
    pyemma.plots.plot_contour(
        *tica_concatenated[:, :2].T,
        eigvec[dtrajs_concatenated, i + 1],
        ax=ax,
        cmap='PiYG',
        cbar_label='{}. right eigenvector'.format(i + 2),
        mask=True)
    ax.set_xlabel('IC 1')
axes[0].set_ylabel('IC 2')
fig.tight_layout()

## 5. Coarse-graining into metastable states

### Perron cluster cluster analysis

The PCCA++ algorithm computes so called memberships, i.e., the probability of each microstate to belong to a given macrostate.
In other words, PCCA++ does a fuzzy assignment of the microstates to macrostates which is encoded in the memberships.
We can visualize the $5$ membership distributions over the first $2$ TICA dimensions as follows:

In [ ]:

nstates = 4

msm.pcca(nstates)
metastable_traj = msm.metastable_assignments[dtrajs_concatenated]

fig, ax = plt.subplots(figsize=(5, 4))
_, _, misc = pyemma.plots.plot_state_map(
    *tica_concatenated[:, :2].T, metastable_traj, ax=ax)
ax.set_xlabel('IC 1')
ax.set_ylabel('IC 2')
misc['cbar'].set_ticklabels([r'$\mathcal{S}_%d$' % (i + 1)
                             for i in range(nstates)])
fig.tight_layout()


<font color='red'>

**QUESTIONS**  
</font>

1. Has PCCA procedure separated the state space within the first two TICA components ?
2. Run the next cell to write pdb files containing example of structure of the different states, and investigate them using VMD.  Are the states well defined  ? 


In [ ]:
# import os

# # Ensure the data directory exists
# os.makedirs('./data', exist_ok=True)

pcca_samples = msm.sample_by_distributions(msm.metastable_distributions, 10)
torsions_source = pyemma.coordinates.source(files, features=torsions_feat)
outfiles = ['./data/pcca{}_10samples.pdb'.format(n + 1) for n in range(msm.n_metastable)]
pyemma.coordinates.save_trajs(
    torsions_source,
    pcca_samples,
    outfiles=outfiles)

print(outfiles)

# visualise the peptide metastable states
Vizualize = True
if Vizualize : 
    vmd = pexpect.spawn("/usr/local/bin/vmd-1.9.4a57") # Ou "/chemin/vers/le/programme"
    vmd.expect("vmd > ") # Le paramètre est le prompt du programme
    for of in outfiles :
        vmd.sendline(f"mol load pdb {of}")
        vmd.expect("vmd > ")

### Chapman-Kolmogorov test
The model is validated with a Chapman-Kolmogorov test. It compares the right and the left side of the Chapman-Kolmogorov equation

$$ \mathbf{P}(k \tau) = \mathbf{P}^k(\tau) $$

with $\mathbf{P}(\tau)$ being the transition matrix and lag time $\tau$.
The highest $k$ can be adjusted using the `mlags` keyword argument of `msm.cktest()`.

<font color='red'>

**QUESTIONS**  
</font>

1.  Run the next cell to perform the CK test.  Is the model Markovian  ? 
2. rerun with smaller or higher number of states  and see the difference . Is there a single MSM than maps the data ?


In [ ]:
cktest = msm.cktest(nstates, mlags=8)
pyemma.plots.plot_cktest(cktest, dt=0.1, units='ns');

This coarse-grained representation of the dynamics is more directly amenable to human interpretation.
Nevertheless, as for the conventional MSM, we can still compute several interesting properties.
We start with the stationary distribution which encodes the free energy of the states.
This can be achieved by summing all the  contributions to a coarse-grained state $\mathcal{S}_i$:

$$ G_i = - \textrm{k}_\textrm{B} T \ln \sum_{j\in \mathcal{S}_i} \pi_j $$

In [ ]:
print('state\tπ\t\tG/kT')
for i, s in enumerate(msm.metastable_sets):
    p = msm.pi[s].sum()
    print('{}\t{:f}\t{:f}'.format(i + 1, p, -np.log(p)))


<font color='red'>

**QUESTIONS**  
</font>

1. Which is the most stable state ?
2. Which  states are the most unstable states ?


Knowing PCCA++ metastable states, we can extract information on the kinetics.
In particular, the mean first passage times (MFPTs) between them:

In [ ]:

mfpt = np.zeros((nstates, nstates))
for i, j in product(range(nstates), repeat=2):
    mfpt[i, j] = msm.mfpt(
        msm.metastable_sets[i],
        msm.metastable_sets[j])

print('Mean First Passage Time in ns:')

str = " initial \ final      "
for j in range(nstates) :
        str = str + f"{j}\t\t"
print(str)

for i in range(nstates) :
    str = ""
    for j in range(nstates) :
        str = str + f"{np.round(mfpt[i, j],2):6.2f}  \t"
    print(f"{i}                  {str}")





<font color='red'>

**QUESTIONS**  
</font>

1. What is the MFPT to go from the less stable state to the most stable state ?
2. What is the MFPT  for the reverse  ? 
3. Could this transition be observed in the trajectories of 500 ns ?
